[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/09.%20The%20Quay%20Crane%20Scheduling%20Problem/P9-Tier-3_executed.ipynb)

---


# 9. The Quay Crane Scheduling Problem
## Tier 3 — Genetic Algorithm

This notebook applies a Genetic Algorithm (GA) to a quay crane scheduling problem (QCSP). The GA maintains a population of candidate assignments and repeatedly uses selection, crossover, mutation, and elitism to search for a schedule with a small makespan. It is a stochastic heuristic: it can find strong solutions, but it does not certify that a solution is globally optimal.

### Learning goals
- Explain how a chromosome represents a crane assignment for every bay.
- Calculate schedule makespan and convert it into a fitness value.
- Describe tournament selection, one-point crossover, mutation, and elitism.
- Interpret a convergence plot and a mutation-rate sensitivity experiment.
- Compare the GA result with the Tier 2 Enhanced LPT heuristic on the same instance and under the same schedule evaluation model.

### Inputs and outputs
The input is a fixed set of bays with processing times and positions, together with three cranes and a travel-time rate. The notebook returns a best-found crane assignment, its schedule and makespan, convergence and sensitivity visualizations, and a comparison with the Tier 2 heuristic.

### Scope and model assumptions
Each bay is processed exactly once by one crane. A chromosome determines which crane receives each bay; within a crane's assigned bays, the service order is fixed in increasing bay position. The initial positioning of each crane at its first assigned bay has zero travel time. Travel between consecutive assigned bays is charged at a constant rate per bay-position unit. Cranes are treated as independent: interference, crossing restrictions, precedence constraints, setup times, and uncertainty in processing or travel times are outside this instance. These assumptions define the schedule evaluated by both the GA and the Tier 2 comparison.

In [ ]:
# Dependencies are expected to be available in the notebook environment.
# Install packages in the environment, rather than from this notebook, if needed.

try:
    import random
    from dataclasses import dataclass
    from typing import Dict, List, Tuple

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from matplotlib.patches import Patch
except ImportError as error:
    raise ImportError(
        "Missing dependency. Install numpy, pandas, and matplotlib in your environment."
    ) from error

print("Dependencies imported successfully.")

## Concrete QCSP instance (10 bays, 3 cranes)

The instance has ten bays, three cranes, and the following processing times:

| Bay | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Processing time (minutes) | 45 | 30 | 25 | 35 | 40 | 20 | 55 | 28 | 32 | 38 |

Bay positions are 1 through 10, and the travel time between positions \(p\) and \(q\) is \(5|p-q|\) minutes. Crane IDs are 0, 1, and 2.

### Chromosome representation
A chromosome is a list of ten crane IDs. Its entry at index \(i\) assigns the corresponding bay in `bay_ids` to one crane. For example, a gene value of `2` means that the corresponding bay is assigned to crane 2. This representation searches over assignments, not arbitrary service orders; the service order is determined by the increasing-position assumption stated above.

### Genetic Algorithm components
- **Fitness:** the negative makespan, so a smaller makespan gives a larger fitness.
- **Selection:** tournament selection samples candidate individuals and chooses the one with the highest fitness.
- **Crossover:** one-point crossover exchanges the portions of two parent chromosomes after a randomly chosen cut.
- **Mutation:** each gene independently changes crane with probability `mut_rate`.
- **Elitism:** the best individuals are copied directly into the next population, preserving their assignments.

In [ ]:
# Data model for the 10-bay, 3-crane instance
@dataclass(frozen=True)
class Bay:
    id: int
    position: int
    process_time: int


bays: List[Bay] = [
    Bay(id=1, position=1, process_time=45),
    Bay(id=2, position=2, process_time=30),
    Bay(id=3, position=3, process_time=25),
    Bay(id=4, position=4, process_time=35),
    Bay(id=5, position=5, process_time=40),
    Bay(id=6, position=6, process_time=20),
    Bay(id=7, position=7, process_time=55),
    Bay(id=8, position=8, process_time=28),
    Bay(id=9, position=9, process_time=32),
    Bay(id=10, position=10, process_time=38),
]

NUM_CRANES = 3
CRANE_IDS = list(range(NUM_CRANES))
TRAVEL_PER_BAY = 5.0

bay_by_id: Dict[int, Bay] = {bay.id: bay for bay in bays}
bay_ids: List[int] = [bay.id for bay in bays]
positions: Dict[int, int] = {bay.id: bay.position for bay in bays}
process_times: Dict[int, int] = {
    bay.id: bay.process_time for bay in bays
}

assert len(bay_ids) == len(set(bay_ids))
assert set(positions) == set(bay_ids)
assert set(process_times) == set(bay_ids)

bays, positions, process_times

## Step 1 — GA helper functions

The first step is to evaluate assignments and implement the operations used by the GA. A schedule's **makespan** is the completion time of its last-finishing crane. For each crane, the evaluation adds the processing times of its assigned bays and the travel times between those bays in increasing position order. The initial trip to the first assigned bay is not charged.

Let \(B\) be the set of bays and \(C\) the set of cranes. A chromosome assigns each bay \(b\) to exactly one crane \(z_b\in C\). Let \(S_c\) be the bays assigned to crane \(c\), written in increasing position order as \(b_{c,1},\ldots,b_{c,m_c}\). The completion time of crane \(c\) is

$$\
T_c = \begin{cases}
0, & m_c=0,\\
\displaystyle\sum_{j=1}^{m_c} p_{b_{c,j}}
+ \tau\sum_{j=2}^{m_c}
\left|x_{b_{c,j}}-x_{b_{c,j-1}}\right|, & m_c>0,
\end{cases}
\qquad
C_{\max}=\max_{c\in C}T_c,
$$

where \(p_b\) is bay \(b\)'s processing time, \(x_b\) is its position, and \(\tau\) is travel time per position unit. The fitness is \(f=-C_{\max}\), so maximizing fitness is equivalent to minimizing makespan. In the code, `makespan_for_chromosome` evaluates \(C_{\max}\), and `fitness` returns \(f\).

An empty crane sequence contributes zero completion time. For a nonempty sequence, the first bay contributes processing time but no initial travel. Sorting assigned bays by position implements the fixed service-order assumption. Each evaluation sorts the bays assigned to each crane, so its time is \(O(|B|\log |B|)\) in the general case.

In [ ]:
def travel_time(
    pos_from: int,
    pos_to: int,
    travel_per_bay: float,
) -> float:
    """Return travel time between two bay positions, in minutes."""
    return travel_per_bay * abs(pos_from - pos_to)


def makespan_for_chromosome(
    chromosome: List[int],
    bay_ids: List[int],
    positions: Dict[int, int],
    process_times: Dict[int, int],
    crane_ids: List[int],
    travel_per_bay: float,
) -> float:
    """Evaluate the makespan of a bay-to-crane assignment chromosome."""
    if len(chromosome) != len(bay_ids):
        raise ValueError("The chromosome must contain one gene per bay.")
    if not set(chromosome).issubset(set(crane_ids)):
        raise ValueError("Every chromosome gene must identify a crane.")
    if set(bay_ids) - set(positions) or set(bay_ids) - set(process_times):
        raise ValueError("Every bay must have a position and processing time.")

    crane_sequences: Dict[int, List[int]] = {
        crane_id: [] for crane_id in crane_ids
    }
    for bay_id, crane_id in zip(bay_ids, chromosome):
        crane_sequences[crane_id].append(bay_id)

    for crane_id in crane_ids:
        crane_sequences[crane_id].sort(key=lambda bay_id: positions[bay_id])

    def crane_completion(sequence: List[int]) -> float:
        if not sequence:
            return 0.0

        completion_time = 0.0
        current_position = positions[sequence[0]]
        for bay_id in sequence:
            bay_position = positions[bay_id]
            completion_time += travel_time(
                current_position, bay_position, travel_per_bay
            )
            completion_time += process_times[bay_id]
            current_position = bay_position
        return completion_time

    return max(
        crane_completion(crane_sequences[crane_id])
        for crane_id in crane_ids
    )


def fitness(
    chromosome: List[int],
    bay_ids: List[int],
    positions: Dict[int, int],
    process_times: Dict[int, int],
    crane_ids: List[int],
    travel_per_bay: float,
) -> float:
    """Return negative makespan, so higher fitness means a better schedule."""
    return -makespan_for_chromosome(
        chromosome,
        bay_ids,
        positions,
        process_times,
        crane_ids,
        travel_per_bay,
    )


def tournament_selection(
    pop: List[List[int]],
    fitnesses: List[float],
    k: int = 2,
) -> List[int]:
    """Choose the fittest individual among k randomly sampled individuals."""
    if len(pop) != len(fitnesses):
        raise ValueError("Population and fitness lists must have equal length.")
    if not 1 <= k <= len(pop):
        raise ValueError("Tournament size must be between 1 and population size.")

    indices = random.sample(range(len(pop)), k)
    best_index = max(indices, key=lambda index: fitnesses[index])
    return pop[best_index].copy()


def one_point_crossover(
    parent1: List[int],
    parent2: List[int],
) -> Tuple[List[int], List[int]]:
    """Create two children by exchanging parent tails after one cut point."""
    if len(parent1) != len(parent2):
        raise ValueError("Parents must have the same chromosome length.")
    if len(parent1) <= 1:
        return parent1.copy(), parent2.copy()

    cut = random.randint(1, len(parent1) - 1)
    child1 = parent1[:cut] + parent2[cut:]
    child2 = parent2[:cut] + parent1[cut:]
    return child1, child2


def mutate(
    chromosome: List[int],
    mut_rate: float,
    crane_ids: List[int],
) -> List[int]:
    """Independently reassign each selected gene to a different crane."""
    if not 0.0 <= mut_rate <= 1.0:
        raise ValueError("Mutation rate must be between 0 and 1.")
    if len(crane_ids) < 2:
        raise ValueError("Mutation requires at least two cranes.")
    if not set(chromosome).issubset(set(crane_ids)):
        raise ValueError("Every chromosome gene must identify a crane.")

    mutated = chromosome.copy()
    for index, current_crane in enumerate(mutated):
        if random.random() < mut_rate:
            alternatives = [
                crane_id
                for crane_id in crane_ids
                if crane_id != current_crane
            ]
            mutated[index] = random.choice(alternatives)
    return mutated


# A sample chromosome illustrates the evaluation without asserting an outcome.
sample_chromosome = [random.choice(CRANE_IDS) for _ in bay_ids]
sample_makespan = makespan_for_chromosome(
    sample_chromosome,
    bay_ids,
    positions,
    process_times,
    CRANE_IDS,
    TRAVEL_PER_BAY,
)
sample_fitness = fitness(
    sample_chromosome,
    bay_ids,
    positions,
    process_times,
    CRANE_IDS,
    TRAVEL_PER_BAY,
)
assert sample_fitness == -sample_makespan
print("Sample chromosome:", sample_chromosome)
print("Makespan (minutes):", sample_makespan)
print("Fitness:", sample_fitness)

## Step 2 — GA main loop

The GA begins with a randomly generated population. In each generation, it evaluates every chromosome, records the best and average makespan, copies the elite individuals, and fills the rest of the next population using tournament selection, crossover, and mutation. The best solution found across all generations is retained.

The population size is \(P\), the number of generations is \(G\), and each chromosome has \(|B|\) genes. For this implementation, evaluation and sorting of the population dominate the work; a useful upper-order description is \(O(GP(|B|\log |B|+\log P))\). This describes computational effort, not a guarantee of solution quality. The search space contains \(3^{10}\) assignments for this instance, while the GA samples and evolves only a portion of that space.

A fixed random seed makes a run reproducible in the same software environment. The algorithm remains stochastic in its search operations, and a different seed can produce a different result.

In [ ]:
def run_ga(
    bay_ids: List[int],
    positions: Dict[int, int],
    process_times: Dict[int, int],
    crane_ids: List[int],
    travel_per_bay: float,
    pop_size: int = 50,
    generations: int = 200,
    mut_rate: float = 0.05,
    elite_size: int = 2,
    seed: int = 42,
) -> Dict:
    """Run the Genetic Algorithm and return its best solution and history."""
    if pop_size < 3:
        raise ValueError("Population size must be at least 3 for tournament selection.")
    if generations < 1:
        raise ValueError("The number of generations must be positive.")
    if not 0 <= elite_size <= pop_size:
        raise ValueError("Elite size must be between 0 and population size.")
    if not 0.0 <= mut_rate <= 1.0:
        raise ValueError("Mutation rate must be between 0 and 1.")
    if not bay_ids or not crane_ids:
        raise ValueError("The instance must contain bays and cranes.")

    random.seed(seed)
    np.random.seed(seed)

    population = [
        [random.choice(crane_ids) for _ in bay_ids]
        for _ in range(pop_size)
    ]

    best_makespan_per_gen = []
    avg_makespan_per_gen = []
    best_chromosome = None
    best_makespan = float("inf")

    for _ in range(generations):
        fitnesses = [
            fitness(
                individual,
                bay_ids,
                positions,
                process_times,
                crane_ids,
                travel_per_bay,
            )
            for individual in population
        ]
        makespans = [-individual_fitness for individual_fitness in fitnesses]

        generation_best = min(makespans)
        generation_average = sum(makespans) / len(makespans)
        best_makespan_per_gen.append(generation_best)
        avg_makespan_per_gen.append(generation_average)

        if generation_best < best_makespan:
            best_makespan = generation_best
            best_index = makespans.index(generation_best)
            best_chromosome = population[best_index].copy()

        # Elitism preserves the strongest assignments while the rest of the
        # population continues to explore through selection and variation.
        sorted_indices = sorted(
            range(len(population)),
            key=lambda index: fitnesses[index],
            reverse=True,
        )
        next_population = [
            population[index].copy()
            for index in sorted_indices[:elite_size]
        ]

        while len(next_population) < pop_size:
            parent1 = tournament_selection(population, fitnesses, k=3)
            parent2 = tournament_selection(population, fitnesses, k=3)
            child1, child2 = one_point_crossover(parent1, parent2)
            child1 = mutate(child1, mut_rate, crane_ids)
            child2 = mutate(child2, mut_rate, crane_ids)
            next_population.append(child1)
            if len(next_population) < pop_size:
                next_population.append(child2)

        population = next_population

    return {
        "best_chromosome": best_chromosome,
        "best_makespan": best_makespan,
        "best_makespan_per_gen": best_makespan_per_gen,
        "avg_makespan_per_gen": avg_makespan_per_gen,
    }


ga_result = run_ga(
    bay_ids=bay_ids,
    positions=positions,
    process_times=process_times,
    crane_ids=CRANE_IDS,
    travel_per_bay=TRAVEL_PER_BAY,
    pop_size=50,
    generations=200,
    mut_rate=0.05,
    elite_size=2,
    seed=42,
)
print(f"GA best makespan (minutes): {ga_result['best_makespan']:.1f}")
print("Best chromosome (crane per bay):", ga_result["best_chromosome"])

## Step 3 — Convergence visualization

The plot shows the best and average makespan in each recorded generation. With elitism, the best value found in the population cannot worsen from one generation to the next: a best individual is carried into the next population. The average can rise or fall because the remaining individuals are replaced through selection and variation. Convergence indicates progress in the recorded search; it does not prove global optimality.

In [ ]:
plt.figure(figsize=(7, 4))
generations = list(range(len(ga_result["best_makespan_per_gen"])))
plt.plot(
    generations,
    ga_result["best_makespan_per_gen"],
    label="Best makespan",
    color="#2E90FA",
)
plt.plot(
    generations,
    ga_result["avg_makespan_per_gen"],
    label="Average makespan",
    color="#12B76A",
    alpha=0.7,
)
plt.xlabel("Generation")
plt.ylabel("Makespan (minutes)")
plt.title("GA convergence for the QCSP (10 bays, 3 cranes)")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## Step 4 — Convert the best chromosome to a readable schedule

A chromosome records assignments but not start and finish times. To make its operational interpretation visible, the next function groups bays by crane, orders each group by position, and calculates travel, processing, start, and finish times using the same assumptions as the fitness evaluation.

In [ ]:
def chromosome_to_schedule(
    chromosome: List[int],
    bay_ids: List[int],
    positions: Dict[int, int],
    process_times: Dict[int, int],
    crane_ids: List[int],
    travel_per_bay: float,
) -> pd.DataFrame:
    """Convert a chromosome into a row-per-bay schedule table."""
    if len(chromosome) != len(bay_ids):
        raise ValueError("The chromosome must contain one gene per bay.")
    if not set(chromosome).issubset(set(crane_ids)):
        raise ValueError("Every chromosome gene must identify a crane.")

    crane_sequences: Dict[int, List[int]] = {
        crane_id: [] for crane_id in crane_ids
    }
    for bay_id, crane_id in zip(bay_ids, chromosome):
        crane_sequences[crane_id].append(bay_id)

    rows = []
    for crane_id, sequence in crane_sequences.items():
        sequence.sort(key=lambda bay_id: positions[bay_id])
        if not sequence:
            continue

        time = 0.0
        current_position = positions[sequence[0]]
        for step_index, bay_id in enumerate(sequence, start=1):
            bay_position = positions[bay_id]
            travel = travel_time(
                current_position, bay_position, travel_per_bay
            )
            time += travel
            start = time
            time += process_times[bay_id]
            finish = time
            rows.append(
                {
                    "crane_id": crane_id,
                    "step": step_index,
                    "bay_id": bay_id,
                    "bay_position": bay_position,
                    "process_time": process_times[bay_id],
                    "travel_time": travel,
                    "start": start,
                    "finish": finish,
                }
            )
            current_position = bay_position

    return pd.DataFrame(rows).sort_values(
        ["crane_id", "start"]
    ).reset_index(drop=True)


ga_schedule_df = chromosome_to_schedule(
    ga_result["best_chromosome"],
    bay_ids,
    positions,
    process_times,
    CRANE_IDS,
    TRAVEL_PER_BAY,
)
ga_schedule_df

In [ ]:
# Recompute completion times from the displayed schedule as an independent check.
assert ga_schedule_df["bay_id"].is_unique
assert set(ga_schedule_df["bay_id"]) == set(bay_ids)

grouped_completion = (
    ga_schedule_df.groupby("crane_id")["finish"].max().to_dict()
)
crane_completion_from_table = {
    crane_id: grouped_completion.get(crane_id, 0.0)
    for crane_id in CRANE_IDS
}
recomputed_makespan = max(crane_completion_from_table.values())
assert np.isclose(
    recomputed_makespan,
    ga_result["best_makespan"],
    rtol=1e-9,
    atol=1e-9,
)

print("Per-crane completion times (minutes):", crane_completion_from_table)
print(f"Recomputed makespan (minutes): {recomputed_makespan:.1f}")
print("Schedule and GA makespans agree.")

## Step 5 — Gantt visualization of the GA schedule

Each bar represents bay processing on one crane; its horizontal position and length show the processing start time and duration. Travel time is included in the schedule calculations and appears as a gap before the next processing bar. The initial positioning assumption contributes no travel time.

In [ ]:
def plot_gantt(schedule: pd.DataFrame, title: str = "") -> None:
    """Plot bay processing intervals on a Gantt-style chart."""
    if schedule.empty:
        print("No schedule to plot.")
        return

    fig, ax = plt.subplots(figsize=(8, 3.5))
    colors = {0: "#2E90FA", 1: "#12B76A", 2: "#F97316"}
    y_ticks = []
    y_tick_labels = []

    for crane_id in sorted(schedule["crane_id"].unique()):
        crane_rows = schedule[schedule["crane_id"] == crane_id]
        y_position = float(crane_id)
        y_ticks.append(y_position)
        y_tick_labels.append(f"Crane {crane_id}")

        for _, row in crane_rows.iterrows():
            start = row["start"]
            finish = row["finish"]
            duration = finish - start
            bay_id = int(row["bay_id"])

            ax.barh(
                y=y_position,
                width=duration,
                left=start,
                height=0.4,
                color=colors.get(crane_id, "#667085"),
                edgecolor="#101828",
                alpha=0.85,
            )
            ax.text(
                start + duration / 2.0,
                y_position,
                str(bay_id),
                ha="center",
                va="center",
                fontsize=9,
                color="white",
            )

    legend_handles = [
        Patch(
            facecolor=colors[crane_id],
            edgecolor="#101828",
            label=f"Crane {crane_id}",
        )
        for crane_id in sorted(schedule["crane_id"].unique())
    ]
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_tick_labels)
    ax.set_xlabel("Time (minutes)")
    ax.set_title(title or "QCSP schedule")
    ax.legend(handles=legend_handles, title="Crane")
    ax.grid(True, axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()


plot_gantt(
    ga_schedule_df,
    title="Tier 3 QCSP — Genetic Algorithm schedule",
)

## Step 6 — Comparison with the Tier 2 heuristic

The Tier 2 Enhanced LPT heuristic assigns bays in decreasing processing-time order to the crane with the smallest current provisional completion time. Its provisional crane load is updated in assignment order; after all assignments are made, each crane's bays are sorted by position and the final makespan is evaluated using the same schedule assumptions as the GA. The comparison is therefore between two assignment methods under one common makespan definition.

The relative change is computed as

$$\
100\times\frac{C_{\max}^{\mathrm{LPT}}-C_{\max}^{\mathrm{GA}}}
{C_{\max}^{\mathrm{LPT}}}\%.
$$

A positive value means the GA's makespan is smaller for these runs; a negative value means it is larger. This single-instance comparison is not an optimality guarantee or a statistical performance study.

In [ ]:
def enhanced_lpt_heuristic_10bay(
    bay_ids: List[int],
    positions: Dict[int, int],
    process_times: Dict[int, int],
    crane_ids: List[int],
    travel_per_bay: float,
) -> Dict:
    """Assign longest-processing-time bays to the least-loaded crane."""
    sorted_bays = sorted(
        bay_ids,
        key=lambda bay_id: process_times[bay_id],
        reverse=True,
    )
    crane_assignments: Dict[int, List[int]] = {
        crane_id: [] for crane_id in crane_ids
    }
    crane_completion: Dict[int, float] = {
        crane_id: 0.0 for crane_id in crane_ids
    }

    for bay_id in sorted_bays:
        best_crane = min(
            crane_ids,
            key=lambda crane_id: crane_completion[crane_id],
        )
        crane_assignments[best_crane].append(bay_id)

        # The heuristic updates its provisional load in assignment order.
        sequence = crane_assignments[best_crane]
        time = 0.0
        current_position = positions[sequence[0]]
        for assigned_bay in sequence:
            bay_position = positions[assigned_bay]
            time += travel_time(
                current_position, bay_position, travel_per_bay
            )
            time += process_times[assigned_bay]
            current_position = bay_position
        crane_completion[best_crane] = time

    # Final evaluation uses the model's increasing-position service order.
    for crane_id in crane_ids:
        crane_assignments[crane_id].sort(
            key=lambda bay_id: positions[bay_id]
        )
        sequence = crane_assignments[crane_id]
        time = 0.0
        if sequence:
            current_position = positions[sequence[0]]
            for bay_id in sequence:
                bay_position = positions[bay_id]
                time += travel_time(
                    current_position, bay_position, travel_per_bay
                )
                time += process_times[bay_id]
                current_position = bay_position
        crane_completion[crane_id] = time

    makespan = max(crane_completion.values())
    return {
        "makespan": makespan,
        "assignments": crane_assignments,
        "completion_times": crane_completion,
    }


heuristic_10bay = enhanced_lpt_heuristic_10bay(
    bay_ids,
    positions,
    process_times,
    CRANE_IDS,
    TRAVEL_PER_BAY,
)
print(
    "Tier 2 heuristic makespan (minutes):",
    heuristic_10bay["makespan"],
)
print("Tier 3 GA makespan (minutes):", ga_result["best_makespan"])

improvement_pct = (
    (heuristic_10bay["makespan"] - ga_result["best_makespan"])
    / heuristic_10bay["makespan"]
) * 100
print(f"Relative change versus Tier 2 heuristic: {improvement_pct:.1f}%")

## Step 7 — Sensitivity: effect of mutation rate

Mutation changes assignments and therefore helps the population explore different parts of the chromosome search space. A very low rate may produce little variation, while a high rate may disrupt useful inherited assignments. These are tendencies, not universal rules: their effects depend on the instance, the other GA settings, and the random search path.

The experiment runs the same population size, generation count, and seed for mutation rates 0.01, 0.05, and 0.1. Each rate is represented by one run, so the comparison illustrates sensitivity for these runs rather than estimating average performance across random seeds.

In [ ]:
def run_ga_with_mut_rate(
    mut_rate: float,
    seed: int = 42,
) -> Dict:
    """Run the instance with one mutation rate and return its best makespan."""
    result = run_ga(
        bay_ids=bay_ids,
        positions=positions,
        process_times=process_times,
        crane_ids=CRANE_IDS,
        travel_per_bay=TRAVEL_PER_BAY,
        pop_size=50,
        generations=200,
        mut_rate=mut_rate,
        elite_size=2,
        seed=seed,
    )
    return {
        "mut_rate": mut_rate,
        "best_makespan": result["best_makespan"],
    }


mut_rates = [0.01, 0.05, 0.1]
rows = [run_ga_with_mut_rate(rate, seed=42) for rate in mut_rates]
mut_sensitivity_df = pd.DataFrame(rows)
mut_sensitivity_df["best_makespan"] = (
    mut_sensitivity_df["best_makespan"].round(1)
)
mut_sensitivity_df

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.bar(
    [str(rate) for rate in mut_sensitivity_df["mut_rate"]],
    mut_sensitivity_df["best_makespan"],
    color="#2E90FA",
    edgecolor="#101828",
    alpha=0.85,
)
plt.xlabel("Mutation rate")
plt.ylabel("Best makespan (minutes)")
plt.title("Mutation-rate sensitivity (10 bays, 3 cranes)")
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Step 8 — Why this Tier exists alongside Tier 2

### Strengths and limitations
- **Population-based search:** the GA evolves several assignments rather than making a single sequence of greedy assignment decisions.
- **Flexible variation:** crossover and mutation provide mechanisms for combining and changing assignments. Their usefulness depends on the chromosome representation and the model being optimized.
- **No optimality certificate:** a good convergence curve or a smaller makespan than a heuristic does not establish that the global optimum has been found.
- **Computational effort and parameter choices:** evaluating a population over many generations requires more fitness evaluations than a simple heuristic, and results can depend on population size, generation count, mutation rate, elitism, and random seed.
- **Stochastic variability:** a single seeded run is reproducible in the same environment, but other seeds may yield different solutions. The sensitivity experiment here is illustrative, not a multi-run statistical analysis.

### When this approach is useful
A GA is useful when a population-based search is appropriate and there is sufficient computational budget for repeated schedule evaluations. Its assignment representation can be extended when the model is extended, but new operational constraints must also be represented and enforced in the evaluation and search operators.

### Connection to later topics
Later topics may study adaptive dispatching policies, real-time data, or coordination among interacting systems. Those topics address broader modeling and decision-making settings; they are not implemented in this notebook. Here, the focus is the GA search procedure and its comparison with the Tier 2 heuristic on one fixed instance.